[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/practicas/Practica_5b.ipynb)

# Práctica 5b: Estimación del ingreso del hogar
## Enfoque en predicción y selección de variables

## Objetivo de aprendizaje
Construir e interpretar un modelo de regresión lineal múltiple, en particular:
- La selección y justificación de las variables independientes
- La R cuadrada y la R cuadrada ajustada como medidas de ajuste, y su uso para comparar modelos
- La significancia global del modelo y la de cada coeficiente
- La interpretación del signo y la magnitud de los coeficientes
- La distinción entre un modelo que predice y un modelo que explica
- La evaluación de los supuestos del modelo

## Estructura del ejercicio
Esta práctica es una alternativa a la Práctica 5 (satisfacción del cliente), orientada a la estimación de ingresos. Ambas cubren el mismo contenido estadístico, por lo que **debes elegir solo una de las dos**.

En la sección de "Contexto y preparación de datos" únicamente debes
- Sustituir la entidad federativa por una diferente a la del ejemplo (19, Nuevo León)

En la sección de "Análisis e interpretación":
- Es donde debes decidir qué variables entran al modelo y escribir tu código
- Es donde debes realizar la interpretación estadística y de negocio del modelo

Como en la Práctica 5, aquí **no se te indica qué variables usar**. Elegirlas y defender esa elección es la parte central de la actividad.

## Contexto y preparación de datos
Supongamos que trabajas en una institución financiera que quiere colocar crédito entre trabajadores sin ingreso comprobable. En México, una parte muy grande de la población económicamente activa trabaja en la informalidad: no tiene recibos de nómina ni declaraciones que respalden lo que gana. Negar el crédito a todos ellos deja fuera a un mercado enorme; otorgarlo sin ninguna estimación de su capacidad de pago es imprudente.

La alternativa que se usa en la práctica, tanto en instituciones financieras como en los programas sociales que necesitan focalizar apoyos, es estimar el ingreso a partir de **indicadores observables**: cuánto gasta el hogar en categorías específicas y cuáles son sus características demográficas. A estos modelos se les conoce como *proxy means tests*.

Tu tarea es construir ese modelo: **estimar el ingreso corriente del hogar en función de su gasto por categoría y de sus características**, para una entidad federativa.

Utilizaremos el archivo `enigh2025.xlsx`, que contiene información de 91,414 hogares de todo el país, obtenida de la Encuesta Nacional de Ingresos y Gastos de los Hogares (ENIGH) 2025 del INEGI. El archivo está localizado en: https://github.com/adan-rs/amd/raw/main/data/enigh2025.xlsx

El archivo pesa alrededor de 21 MB, por lo que la descarga y la lectura pueden tardar algunos segundos.

**Variables de identificación y características del hogar**

|VARIABLE|DESCRIPCIÓN|VALORES|
|---|---|---|
|`ubica_geo`|Clave de entidad y municipio|Los dos primeros dígitos son la entidad federativa|
|`tam_loc`|Tamaño de localidad|1 = 100,000 y más habitantes; 2 = 15,000 a 99,999; 3 = 2,500 a 14,999; 4 = menos de 2,500|
|`est_socio`|Estrato socioeconómico|1 = bajo; 2 = medio bajo; 3 = medio alto; 4 = alto|
|`clase_hog`|Clase de hogar|1 = unipersonal; 2 = nuclear; 3 = ampliado; 4 = compuesto; 5 = corresidente|
|`sexo_jefe`|Sexo del jefe del hogar|1 = hombre; 2 = mujer|
|`edad_jefe`|Edad del jefe del hogar|Años cumplidos (14 a 106)|
|`educa_jefe`|Escolaridad del jefe del hogar|1 = sin instrucción, hasta 11 = posgrado|
|`tot_integ`|Número de integrantes del hogar|1 a 20|

**Variables monetarias** (pesos del trimestre de referencia)

|VARIABLE|DESCRIPCIÓN|
|---|---|
|`ing_cor`|Ingreso corriente total del hogar (**variable dependiente**)|
|`gasto_mon`|Gasto monetario total del hogar|

**Rubros de gasto**
- *Alimentos y bebidas*: `alimentos`, `ali_dentro`, `cereales`, `carnes`, `leche`, `huevo`, `verduras`, `frutas`, `bebidas`, `ali_fuera`
- *Vestido y calzado*: `vesti_calz`, `vestido`, `calzado`
- *Vivienda y servicios*: `vivienda`, `agua`, `energia`, `limpieza`
- *Salud*: `salud`
- *Transporte y comunicaciones*: `transporte`, `publico`, `combus`, `comunica`
- *Educación y esparcimiento*: `educacion`, `esparci`, `paq_turist`
- *Cuidados personales*: `personales`, `cuida_pers`

> **Advertencia: el modelo requiere tu criterio, no el de la computadora.** Que una variable esté en la base de datos no significa que pueda entrar al modelo. Antes de estimar, revisa una por una las variables disponibles y decide si tiene sentido usarlas para estimar el ingreso. Al menos tres situaciones distintas exigen esa revisión:
>
> - **Variables que son la suma de otras.** `gasto_mon` es el gasto total del hogar, es decir, la suma de todos los rubros. Usarlo junto con los rubros introduce la misma información dos veces, y usarlo solo convierte el ejercicio en un atajo contable: en el caso de negocio, un gasto total tampoco es observable ni verificable, que es justo el problema que se quiere resolver. **No lo incluyas como variable independiente.** Además, entre los propios rubros hay **agregados que contienen a otros** de la lista: al menos un par cumple la relación de manera exacta. Identifícalos, decide cuáles conservas y explica en tu análisis qué le pasa a la regresión si se incluyen juntos.
> - **Variables cuyo número no es una cantidad.** `ubica_geo` es una clave geográfica, no una magnitud: sirve para filtrar, no para predecir. Otras variables son categóricas u ordinales codificadas con números (`sexo_jefe`, `clase_hog`, `est_socio`, `tam_loc`, `educa_jefe`) y, si decides usarlas, requieren un tratamiento explícito: variables indicadoras, o una justificación de por qué las tratas como continuas.
> - **La dirección del razonamiento.** El gasto de un hogar no causa su ingreso: ambos se determinan al mismo tiempo. Por eso este es un modelo **predictivo**, no explicativo, y sus coeficientes no son el efecto del gasto sobre el ingreso. Puedes usarlo para estimar el ingreso de un hogar del que solo observas gasto y características, pero no para afirmar que si un hogar gastara más en cierta categoría, ganaría más.
>
> Un modelo con la R cuadrada más alta no es el mejor modelo si sus variables no resisten estas tres preguntas. Se evaluará la justificación de lo que **dejaste fuera** con el mismo peso que lo que incluiste.

> **Nota: alcance didáctico del ejercicio.** El modelo que estimarás aquí es una versión simplificada con fines de aprendizaje. Antes de interpretar los resultados, ten presente que:
> - El archivo es una extracción preparada para el curso y **no incluye el factor de expansión** de la ENIGH: todos los hogares pesan igual, cuando el diseño de la encuesta sobremuestrea ciertos estratos. Por lo tanto los resultados no son estimaciones representativas de la población.
> - La ENIGH, como toda encuesta de ingresos, **subdeclara el ingreso**, sobre todo en los hogares más ricos. Estás modelando el ingreso reportado, no el ingreso real.
> - El ingreso y el gasto se determinan **simultáneamente**. El modelo sirve para predecir, no para atribuir efectos, y los coeficientes no deben leerse como impactos.
> - La unidad de observación es el **hogar**, no el individuo, y las cifras corresponden al periodo de referencia de la encuesta.
> - El modelo se estima **dentro de una sola entidad federativa**, así que sus coeficientes no se trasladan automáticamente a otra.
> - Se evalúa el ajuste sobre los mismos datos con los que se estimó el modelo. Un modelo predictivo real se validaría con datos que no participaron en la estimación.
> - Con miles de observaciones, casi cualquier coeficiente distinto de cero resulta estadísticamente significativo. Distinguir **significancia estadística** de **relevancia práctica** es parte de la interpretación.
>
> Estas limitaciones no invalidan el ejercicio, pero sí acotan las conclusiones que puedes defender.

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

La celda siguiente descarga el archivo, obtiene la entidad federativa a partir de `ubica_geo` y filtra la entidad seleccionada. También conserva únicamente los hogares con ingreso positivo y crea `log_ingreso`.

La transformación logarítmica del ingreso no es un adorno: el ingreso tiene una distribución muy sesgada a la derecha y unos pocos hogares con ingresos muy altos dominarían la estimación en niveles. Trabajar con el logaritmo estabiliza la varianza y hace que los coeficientes se lean como cambios porcentuales aproximados. En tu interpretación deberás decir qué implica esa decisión.

**Utiliza una entidad diferente a la del ejemplo (19, Nuevo León)**: elige cualquier clave del 1 al 32 y justifica tu elección en el análisis.

In [ ]:
# Parámetros
entidad = 19    # Cambiar por otra entidad federativa (clave del 1 al 32)

# Importar el archivo "data/enigh2025.xlsx"
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/enigh2025.xlsx')

# La clave de entidad son los dos primeros dígitos de 'ubica_geo'
df['entidad'] = df['ubica_geo'] // 1000

# Filtrar la entidad seleccionada y conservar hogares con ingreso positivo
datos = df[(df['entidad'] == entidad) & (df['ing_cor'] > 0)].copy()

# Variable dependiente en logaritmo natural
datos['log_ingreso'] = np.log(datos['ing_cor'])

print(f'Hogares en el país:          {len(df)}')
print(f'Hogares en la entidad {entidad}:     {(df["entidad"] == entidad).sum()}')
print(f'Hogares con ingreso positivo: {len(datos)}')

In [ ]:
# Revisar las variables y el número de observaciones
datos.info()

Antes de decidir el modelo conviene ver en qué proporción de hogares el gasto reportado en cada rubro es cero. Un rubro con muchos ceros sigue siendo utilizable como predictor, pero significa que para la mayoría de los hogares esa variable no aporta variación, y conviene tenerlo presente al interpretar su coeficiente.

In [ ]:
# Proporción de hogares con gasto cero en cada rubro
rubros = ['alimentos', 'ali_dentro', 'cereales', 'carnes', 'leche', 'huevo', 'verduras',
          'frutas', 'bebidas', 'ali_fuera', 'vesti_calz', 'vestido', 'calzado', 'vivienda',
          'agua', 'energia', 'limpieza', 'salud', 'transporte', 'publico', 'combus',
          'comunica', 'educacion', 'esparci', 'paq_turist', 'personales', 'cuida_pers']

((datos[rubros] == 0).mean() * 100).round(1).sort_values().to_frame('% hogares con gasto cero')

In [ ]:
# Estadística descriptiva de las características del hogar y del ingreso
caracteristicas = ['tot_integ', 'edad_jefe', 'educa_jefe', 'sexo_jefe',
                   'tam_loc', 'est_socio', 'clase_hog']

datos[caracteristicas + ['ing_cor', 'log_ingreso']].describe().T

## Análisis e interpretación
Construye un modelo para estimar el ingreso del hogar (`log_ingreso`) en función del gasto por rubro y de las características del hogar, para la entidad que seleccionaste.

Tu análisis debe incluir, como mínimo, los siguientes apartados:

**1. Planteamiento del modelo**
- Especifica la variable dependiente.
- Define las variables independientes seleccionadas.

**2. Justificación del modelo**
- Justifica teóricamente o analíticamente la elección de las variables independientes.
- Explica por qué dichas variables permitirían estimar el ingreso de un hogar.
- Justifica también las **exclusiones**: qué variables descartaste y por qué, siguiendo la advertencia de la sección anterior.

**3. Preparación de los datos**
- De ser necesario, haz un manejo apropiado de datos perdidos o atípicos, y documenta el criterio utilizado.
- Si usas variables categóricas, explica cómo las codificaste.
- Si decides transformar algún rubro de gasto, explica cómo resolviste los valores en cero.

**4. Tamaño muestral**
- Verifica que el tamaño de la muestra sea suficiente, considerando el número de variables explicativas utilizadas.

**5. Estadística descriptiva**
- Presenta la estadística descriptiva de las variables cuantitativas utilizadas en el modelo.
- Interpreta brevemente los resultados obtenidos.

**6. Estimación y comparación de modelos**
- Ajusta un modelo de regresión lineal múltiple.
- Estima además, por separado, un modelo con **solo las características del hogar** y otro con **solo el gasto**, y compáralos con el modelo completo.
- Presenta la ecuación estimada del modelo que hayas elegido y explica por qué te quedas con ese.

**7. Evaluación del ajuste del modelo**
- Reporta e interpreta el coeficiente de determinación (R² y R² ajustada).
- Explica por qué al comparar modelos con distinto número de variables conviene mirar la R² ajustada y no la R².

**8. Evaluación de la significancia del modelo**
- Interpreta el p-valor del estadístico F.
- Concluye si el modelo es globalmente significativo.

**9. Evaluación de la significancia de los coeficientes**
- Interpreta los p-valores de los coeficientes.
- Identifica qué variables son estadísticamente significativas y cuáles, aun siéndolo, aportan poco en términos prácticos.

**10. Interpretación de resultados**
- Interpreta el signo y magnitud de los coeficientes, recordando que la variable dependiente está en logaritmo.
- Explica qué indicadores resultan más útiles para estimar el ingreso de un hogar.
- Deriva al menos una implicación para el caso de negocio: ¿el modelo serviría para evaluar la capacidad de pago de un solicitante sin comprobante de ingresos? ¿Con qué reservas?

**11. Evaluación de supuestos del modelo**

Analiza al menos un posible problema relacionado con:
- Linealidad
- Normalidad de los residuos
- Homocedasticidad
- Multicolinealidad
- Independencia de los errores

## Uso de IA generativa (para extender, no para resolver)

La IA se usa para **ampliar** la práctica partiendo de lo que ya construiste, no para hacerla. Declara la herramienta y la versión utilizada (por ejemplo, ChatGPT 5, Claude Opus 4.5, Gemini 3 Pro, Copilot).

**Qué debes entregar** (cuatro bloques, en celdas de texto dentro del notebook):

1. **Tu pregunta de negocio.** Una pregunta propia, pertinente y que la práctica **no** responda. Formúlala en primera persona: "Quiero saber si...", "Me preocupa que...". No se acepta reproducir la pregunta del ejercicio ni preguntas genéricas. Del tipo esperado (no para copiar): *"Quiero saber si mi modelo se equivoca más en los hogares de ingreso bajo que en los de ingreso alto, porque eso decidiría a quién puedo evaluar con él y a quién no"*.
2. **El prompt completo, transcrito en celda de texto** (no de código). Debe incluir: rol, objetivo, tu código, **los resultados reales de tu regresión pegados** (la salida de `summary()`) y una restricción explícita de lo obvio (por ejemplo: "no me expliques qué es la R cuadrada ni qué es la multicolinealidad"). Un prompt de una línea, sin contexto ni resultados, no cuenta.
3. **Qué adopté y qué descarté.** De la respuesta recibida, indica qué implementaste y qué dejaste fuera, con la razón. La implementación usa **máximo 50 líneas de código**.
4. **Verificación.** Un cálculo, contraejemplo o comprobación que valide o refute algo que la IA afirmó. Por ejemplo: si la IA propone agregar una variable que mejora la R cuadrada, verifica que no sea un agregado de otras que ya incluiste; si sugiere que el modelo predice bien, estima con una parte de los hogares y evalúa el error en los hogares restantes.

**Evidencia auditable**: pega el resultado completo de la extensión (texto, tablas o código), no un enlace a la conversación. Revisa que el documento o el código no quede cortado.

**No se acepta**:
- Transferir las instrucciones de la práctica a la IA, ya sea copiándolas o parafraseándolas, para que ella la resuelva.
- Usar la IA como enciclopedia: respuestas generales sin tus datos ni tu código. Está bien usarla para comprender, pero debe haber contenido propio nuevo.
- Transcribir sugerencias sin implementarlas, o dar por cierto lo que la IA afirma sin comprobarlo.
- Delegar la interpretación: las conclusiones y su redacción son tuyas.
- Aceptar una selección automática de variables (por ejemplo, un procedimiento paso a paso sugerido por la IA) sin revisar si las variables elegidas tienen sentido.

**Buenas prácticas sugeridas**: repreguntar a la IA sobre su propia respuesta; pedirle explícitamente las limitaciones de lo que propone; traer un concepto externo al curso y aplicarlo a tus datos.

**Defensa oral**: cualquier práctica puede ser seleccionada al azar para una defensa oral breve (3 a 5 minutos), en la que deberás explicar tus decisiones, tu código y tus conclusiones. Un trabajo que no pueda ser explicado por su autor se considerará evidencia de trabajo no auténtico y podrá ser penalizado.

## Entregable
Notebook en Jupyter exportado a pdf o html, con el código, análisis e interpretación.

## Rúbrica de evaluación
1. **Planteamiento y justificación del modelo (20%)**: se evalúan los puntos 1 y 2. Incluye la justificación de las variables seleccionadas y, con el mismo peso, la de las **excluidas**. Se anula este criterio si el modelo usa `gasto_mon` como predictor o si incluye simultáneamente un rubro agregado y sus componentes sin advertirlo.
2. **Preparación de los datos y tamaño muestral (10%)**: se evalúan los puntos 3 y 4. Tratamiento documentado de datos faltantes y atípicos, codificación explícita de variables categóricas y verificación de que la muestra sostiene el número de predictores.
3. **Análisis descriptivo (10%)**: se evalúa el punto 5. Se penalizará presentar tablas sin interpretación.
4. **Estimación, comparación y ajuste de los modelos (15%)**: se evalúan los puntos 6, 7 y 8. Los tres modelos estimados y comparados, ecuación presentada, e interpretación de R², R² ajustada y estadístico F. Se penalizará reportar los números sin interpretarlos, y elegir el modelo únicamente por tener la R² más alta.
5. **Interpretación de los coeficientes (15%)**: se evalúan los puntos 9 y 10. Interpretación del signo, la magnitud y la significancia, considerando que la dependiente está en logaritmo, con al menos una implicación para el caso de negocio y el señalamiento de que el modelo predice sin explicar. Se penalizará que sólo se indique si el coeficiente es significativo o no.
6. **Evaluación de supuestos (10%)**: se evalúa el punto 11. Diagnóstico realizado con evidencia (gráficos o pruebas) y lectura de lo que implica para la validez del modelo.
7. **Extensión del análisis con IA generativa (20%)**, evaluada en cuatro partes iguales (5% cada una):
   - *Pregunta propia (5%)*: pregunta de negocio formulada por el alumno, pertinente y no respondida por la práctica. Se anula si reproduce la pregunta del ejercicio o si es genérica.
   - *Calidad del prompt (5%)*: transcrito completo en celda de texto, con rol, objetivo, código y los resultados reales de la regresión, más una restricción explícita de lo obvio. Se anula si se transfieren las instrucciones de la práctica (copiadas o parafraseadas).
   - *Ejecución: qué adopté y qué descarté (5%)*: lo propuesto se implementa (máximo 50 líneas de código) y se explica qué quedó fuera y por qué. No basta transcribir sugerencias.
   - *Verificación e interpretación propia (5%)*: un cálculo, contraejemplo o comprobación que valide o refute una afirmación de la IA, y conclusiones redactadas por el alumno.

   *Requisito de forma*: la evidencia debe ser auditable, esto es, resultado completo pegado en el notebook, sin cortes y sin enlaces a la conversación como único respaldo.

La claridad de la redacción y el orden del notebook se consideran de manera transversal: respuestas vagas, confusas o no alineadas con los resultados obtenidos serán penalizadas en el criterio correspondiente.

Se permite el uso de herramientas de apoyo (incluyendo IA) como soporte técnico y, como se indica en la sección anterior, para profundizar en el análisis. Sin embargo, el análisis, la interpretación de resultados y las conclusiones deben reflejar comprensión propia.

Respuestas genéricas, excesivamente uniformes o que no estén alineadas con los resultados obtenidos en el notebook podrán considerarse como evidencia de trabajo no auténtico y serán evaluadas con penalización.